In [1]:
# @title **Data Preparation Driver: the-luau-stack**

# @markdown ---
# @markdown ### **GitHub Repository (Source Code):**
GITHUB_REPO_URL = "https://github.com/bananamort/luau-qwen2.5-coder-1.5b-distillation.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

# @markdown ### **Hugging Face Hub & IO:**
HF_TOKEN = "" # @param {type:"string"}
RAW_DATASET_ID = "TorpedoSoftware/the-luau-stack" # @param {type:"string"}
UPLOAD_DATASET_REPO_ID = "bananamort/the-luau-stack-fim-tokenized" # @param {type:"string"}
OUTPUT_PARQUET_PATH = "fim_train.parquet" # @param {type:"string"}

# @markdown ### **Preprocessing Parameters:**
MAX_SEQ_LEN = 2048 # @param {type:"integer"}
CUTS_PER_FILE = 6 # @param {type:"integer"}
LIMIT = 0 # @param {type:"integer"}

# @markdown ### **Runtime Management:**
AUTO_DISCONNECT_VM = True # @param {type:"boolean"}

import os
import subprocess

def run_cmd_streaming(cmd):
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.stdout.close()
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


if os.path.exists("repo"):
    %cd repo
    !git pull origin {BRANCH}
elif os.path.exists(".git"):
    !git pull origin {BRANCH}
elif GITHUB_REPO_URL.strip():
    print(f"Cloning repository: {GITHUB_REPO_URL} (branch: {BRANCH})...")
    !git clone --depth 1 -b {BRANCH} {GITHUB_REPO_URL.strip()} repo
    %cd repo

!pip install -q pyarrow tokenizers transformers huggingface_hub datasets tqdm

try:
    if HF_TOKEN.strip():
        os.environ["HF_TOKEN"] = HF_TOKEN.strip()

    # 1. Run Data Preparation (Fails fast on any error)
    cmd = [
        "python", "-u", "src/prep_data.py",
        "--dataset_id", RAW_DATASET_ID.strip(),
        "--output_parquet", OUTPUT_PARQUET_PATH.strip(),
        "--max_seq_len", str(MAX_SEQ_LEN),
        "--cuts_per_file", str(CUTS_PER_FILE),
    ]
    if LIMIT > 0:
        cmd.extend(["--limit", str(LIMIT)])
    
    run_cmd_streaming(cmd)

    # 2. Upload Pre-Tokenized Parquet to Hugging Face Dataset Hub
    if HF_TOKEN.strip() and UPLOAD_DATASET_REPO_ID.strip() and os.path.exists(OUTPUT_PARQUET_PATH.strip()):
        from huggingface_hub import HfApi, create_repo
        print(f"Uploading {OUTPUT_PARQUET_PATH} to https://huggingface.co/datasets/{UPLOAD_DATASET_REPO_ID}...")
        api = HfApi(token=HF_TOKEN.strip())
        create_repo(repo_id=UPLOAD_DATASET_REPO_ID.strip(), repo_type="dataset", exist_ok=True)
        api.upload_file(
            path_or_fileobj=OUTPUT_PARQUET_PATH.strip(),
            path_in_repo="fim_train.parquet",
            repo_id=UPLOAD_DATASET_REPO_ID.strip(),
            repo_type="dataset",
        )
        print("Upload complete.")
except Exception as e:
    print(f"Error: {e}")
    raise
finally:
    if AUTO_DISCONNECT_VM:
        print("Done. Unassigning Colab runtime...")
        try:
            from google.colab import runtime
            runtime.unassign()
            print("Successfully disconnected.")
        except Exception as e:
            print(f"Failed to disconnect runtime: {e}")


Cloning repository: https://github.com/bananamort/luau-qwen2.5-coder-1.5b-distillation.git (branch: main)...
Cloning into 'repo'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 50 (delta 8), reused 40 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 850.17 KiB | 12.32 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/repo
Loading dataset from Hugging Face Hub: TorpedoSoftware/the-luau-stack

Generating train split: 100%|██████████| 87978/87978 [00:04<00:00, 17950.66 examples/s]
Loaded 87978 files. Loading tokenizer: TorpedoSoftware/Luau-Qwen3-4B-FIM-v0.1 (cuts_per_file=6)...
Processing with 2 workers...
Progress: 1000/87978 files processed (5964 total FIM samples)
Progress: 2000/87978 files processed (11724 total FIM samples)
Progress: 3000/87978 files processed (17520 total FIM samples)
Progress: 4000/87978 files processed (23406 total FIM samples)
Progress:

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt/repo/fim_train.parquet:   0%|          |  524kB /  248MB            

Upload complete.
Done. Unassigning Colab runtime...
Successfully disconnected.
